In [1]:
import pandas as pd 
from pathlib import Path
import warnings

In [2]:
base_path = Path('/Users/anuj/Documents/CodeBase/Bioinformatics_projects/biological-age-prediction/data/raw')

In [3]:
for year_folder in base_path.iterdir():
    for file in year_folder.glob("*.xpt"):
        print(file.stem.split("_"))

['P', 'HSCRP']
['P', 'DEMO']
['P', 'GLU']
['P', 'CBC']
['P', 'HDL']
['P', 'BIOPRO']
['P', 'GHB']
['BIOPRO', 'G']
['GHB', 'G']
['DEMO', 'G']
['HDL', 'G']
['GLU', 'G']
['CBC', 'G']
['CBC', 'H']
['GLU', 'H']
['HDL', 'H']
['DEMO', 'H']
['GHB', 'H']
['BIOPRO', 'H']
['CBC', 'I']
['GLU', 'I']
['HSV', 'I']
['HDL', 'I']
['DEMO', 'I']
['GHB', 'I']
['BIOPRO', 'I']


In [4]:
df = pd.read_sas(file)

print(df.shape)
print(df.head())

(6744, 38)
      SEQN  LBXSAL  LBDSALSI  LBXSAPSI  LBXSASSI  LBXSATSI  LBXSBU  LBDSBUSI  \
0  83732.0     4.6      46.0      52.0      21.0      25.0    13.0      4.64   
1  83733.0     4.5      45.0      47.0      31.0      35.0    10.0      3.57   
2  83734.0     4.5      45.0      46.0      30.0      29.0    26.0      9.28   
3  83735.0     3.8      38.0      65.0      23.0      26.0    13.0      4.64   
4  83736.0     4.3      43.0      46.0      20.0      13.0    12.0      4.28   

   LBXSC3SI  LBXSCA  ...  LBXSPH  LBDSPHSI  LBXSTB  LBDSTBSI  LBXSTP  \
0      25.0     9.8  ...     4.7     1.518     0.5      8.55     7.5   
1      27.0     9.8  ...     4.4     1.421     0.6     10.26     7.4   
2      24.0     9.7  ...     3.6     1.162     0.5      8.55     7.3   
3      24.0     8.9  ...     3.8     1.227     0.3      5.13     6.1   
4      24.0     9.3  ...     3.2     1.033     0.3      5.13     7.7   

   LBDSTPSI  LBXSTR  LBDSTRSI  LBXSUA  LBDSUASI  
0      75.0   158.0     1

Now - need to code this in 3 functions as to load_data, read_xpt and extract_name.

In [5]:
def extract_name(file: Path) -> str:
    parts = file.stem.split('_')

    if parts[0] == 'P':
        return parts[1]
    return parts[0]

In [6]:
def read_xpt(file: Path) -> pd.DataFrame:
    return pd.read_sas(file, format='xport', encoding= 'latin1')
   

In [7]:
def load_all_data(base_path: Path) -> dict:
    data = {}

    for year_folder in base_path.iterdir():
        if not year_folder.is_dir():
            continue

        year = year_folder.name

        for file in year_folder.glob('*.xpt'):
            dataset_name = extract_name(file)

            try:
                df = read_xpt(file)

                if 'year' not in df.columns:
                    df["year"] = year

                # flat dictionary using tuple key
                key = (year, dataset_name)
                data[key] = df

            except Exception as e:
                print(f'Error reading {file}: {e}')
                continue

    return data


In [8]:
data = load_all_data(base_path)

/var/folders/sl/clykmrq15rd0ksl37m6bnksw0000gn/T/ipykernel_26296/3693786825.py:2: UserWarning: xport file may be corrupted.
  return pd.read_sas(file, format='xport', encoding= 'latin1')


In [9]:
print(len(data))

26


In [10]:
df = list(data.values())[0]

In [11]:
df.shape

(17047, 4)

## Merge_data.py - now will be merging the data

In [12]:
from typing import Tuple, List

def get_metadata(data: dict) -> Tuple[List[str], List[str]]:
    
    years = list(set(k[0] for k in data.keys()))
    features = list(set(k[1] for k in data.keys()))

    return years, features

In [13]:
def yearly_features_merged(data: dict, year: str, features: list) -> pd.DataFrame:


    df_merged = data[(year, 'DEMO')].copy()

    for feature in features:

        if feature == 'DEMO':
            continue

        if (year, feature) not in data:
            continue

        df_feature = data[(year, feature)]

        df_feature = df_feature.drop(columns = ['year'], errors = 'ignore')

        df_merged = pd.merge(df_merged, df_feature, on = "SEQN", how = "left")

    return df_merged
        

In [14]:
def all_years_data(data: dict) -> dict:
    
    years, features = get_metadata(data)
    
    merged_data = {}

    for year in years:
        merged_data[year] = yearly_features_merged(data, year, features)

    return merged_data
        

In [16]:
def combine_years(merged_data: dict)->pd.DataFrame:
    dfs =[]

    for year, df_year in merged_data.items():
        df_year = df_year.copy()
        df_year['year'] = year

        dfs.append(df_year)

    return pd.concat(dfs, ignore_index=True)
    